# LP+Text — Text-Informed Linear Probe
Method from Shakeri et al., "Few-shot Adaptation of Medical Vision-Language Models" (MICCAI 2024, arXiv:2409.03868).

**Why this method:** their benchmark found this simple approach matches or beats far more complex adapters/prompt-learning/LoRA-style methods for few-shot medical VLM adaptation, while being much cheaper and avoiding overfitting on tiny per-class support sets. A companion paper also warns that a full contrastive loss term can hurt generalization specifically in few-shot imbalanced settings -- this method sidesteps that by keeping the text contribution to a simple, low-capacity blend rather than a full contrastive objective.

**Method, plainly:** train a plain linear classifier on frozen image embeddings (like Phase 1), but ADD a small learnable per-class scalar multiplier on how similar the image is to that class's frozen text embedding. Final logit per class = (linear probe's own logit) + (learnable multiplier x cosine similarity to that class's text embedding). Only the linear layer's weights and the per-class multipliers are trained -- everything else (image encoder, text encoder) stays frozen. No deep fusion network, no contrastive loss, no LoRA.

**No leakage risk:** the text similarity is compared against ALL classes for every image (same mechanism as zero-shot) -- the correct label is never handed to the model as an input feature.

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

EMB_PATH = "eyeclip_embeddings.npz"        # Phase 1 frozen image embeddings
SPLIT_DIR = Path("/kaggle/input/datasets/faruq05/rop-vl/ROP-VL/split")  # CHANGE if needed

data = np.load(EMB_PATH)
X_train_img, X_val_img, X_test_img = data["X_train"], data["X_val"], data["X_test"]

train_df = pd.read_csv(SPLIT_DIR / "train.csv")
val_df   = pd.read_csv(SPLIT_DIR / "val.csv")
test_df  = pd.read_csv(SPLIT_DIR / "test.csv")

for name, arr, df in [("X_train", X_train_img, train_df), ("X_val", X_val_img, val_df), ("X_test", X_test_img, test_df)]:
    assert len(arr) == len(df), f"{name} row count != matching split csv"

classes_sorted = sorted(train_df["categories"].unique())
class_to_idx = {c: i for i, c in enumerate(classes_sorted)}
NUM_CLASSES = len(classes_sorted)

y_train_idx = train_df["categories"].map(class_to_idx).values
y_val_idx   = val_df["categories"].map(class_to_idx).values
y_test_idx  = test_df["categories"].map(class_to_idx).values

## Text embeddings (reuse Phase 5's encoding step -- text tower frozen, never fine-tuned)
If you already have `text_embeddings` saved from Phase 5/6c, load it directly instead of re-running EyeCLIP's text encoder here.

In [ ]:
# Option A: reuse a previously saved text embedding file, if you saved one
TEXT_EMB_PATH = "text_embeddings.npz"   # CHANGE or set to None to re-encode from scratch

if TEXT_EMB_PATH is not None:
    try:
        t = np.load(TEXT_EMB_PATH, allow_pickle=True)
        text_embeddings_raw = t["text_embeddings"]      # [8, 512]
        text_class_order = list(t["classes"])
        print("loaded cached text embeddings")
    except FileNotFoundError:
        TEXT_EMB_PATH = None
        print("no cached file found -- will re-encode with EyeCLIP below")

if TEXT_EMB_PATH is None:
    import sys
    sys.path.insert(0, "EyeCLIP")  # CHANGE to your actual path if not already on sys.path
    import eyeclip
    CKPT_PATH = "eyeclip_visual.pt"                          # CHANGE to actual path
    DISEASE_DESC_PATH = "disease_description.xlsx"           # CHANGE to actual path

    text_model, _ = eyeclip.load("ViT-B/32", device=DEVICE, jit=False)
    ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
    state = ckpt.get("model_state_dict") or ckpt.get("state_dict") or ckpt
    text_model.load_state_dict(state, strict=False)
    text_model.eval()

    desc_df = pd.read_excel(DISEASE_DESC_PATH)
    tokens = eyeclip.tokenize(desc_df["description"].tolist(), truncate=True).to(DEVICE)
    with torch.no_grad():
        text_embeddings_raw = text_model.encode_text(tokens).float().cpu().numpy()
    text_class_order = desc_df["disease"].tolist()

assert not (set(classes_sorted) - set(text_class_order)), "class missing a text description"
reorder_idx = [text_class_order.index(c) for c in classes_sorted]
text_embeddings = torch.tensor(np.array(text_embeddings_raw)[reorder_idx], dtype=torch.float32, device=DEVICE)
text_embeddings_norm = F.normalize(text_embeddings, dim=1)  # [NUM_CLASSES, 512], frozen constant
print("text_embeddings shape:", text_embeddings.shape)

## LP+Text model definition
`logits = linear(image_embedding) + multiplier * cosine_similarity(image_embedding, text_embeddings)`
`multiplier` is a single learnable scalar per class (NUM_CLASSES parameters total) -- deliberately low-capacity, per the paper's finding that simple blending outperforms deep fusion in this regime.

In [ ]:
class LPText(nn.Module):
    def __init__(self, img_dim, num_classes, text_embeddings_norm):
        super().__init__()
        self.linear = nn.Linear(img_dim, num_classes)
        self.multiplier = nn.Parameter(torch.ones(num_classes) * 1.0)  # one learnable scalar per class
        self.register_buffer("text_norm", text_embeddings_norm)        # frozen, not trained

    def forward(self, img_emb):
        img_norm = F.normalize(img_emb, dim=1)
        text_sim = img_norm @ self.text_norm.T          # [B, C], cosine similarity to each class's text
        return self.linear(img_emb) + self.multiplier * text_sim

## Full-train evaluation (mirrors Phase 1's setup for direct comparison)

In [ ]:
def to_tensor(x, dtype=torch.float32):
    return torch.tensor(x, dtype=dtype, device=DEVICE)

Xi_train, Xi_val, Xi_test = to_tensor(X_train_img), to_tensor(X_val_img), to_tensor(X_test_img)
Yt_train = to_tensor(y_train_idx, dtype=torch.long)
Yt_val   = to_tensor(y_val_idx,   dtype=torch.long)

IMG_DIM = X_train_img.shape[1]
lp_text_model = LPText(IMG_DIM, NUM_CLASSES, text_embeddings_norm).to(DEVICE)

class_counts = np.bincount(y_train_idx, minlength=NUM_CLASSES)
class_weights = to_tensor(1.0 / np.maximum(class_counts, 1))
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(lp_text_model.parameters(), lr=1e-3, weight_decay=1e-4)

best_val_f1, best_state = -1.0, None
for epoch in range(200):  # linear model -- cheap, can afford more epochs than LoRA training
    lp_text_model.train()
    optimizer.zero_grad()
    logits = lp_text_model(Xi_train)
    loss = criterion(logits, Yt_train)
    loss.backward(); optimizer.step()

    lp_text_model.eval()
    with torch.no_grad():
        val_preds = lp_text_model(Xi_val).argmax(dim=1).cpu().numpy()
    _, _, val_f1, _ = precision_recall_fscore_support(y_val_idx, val_preds, average=None, zero_division=0)
    val_macro_f1 = val_f1.mean()
    if val_macro_f1 > best_val_f1:
        best_val_f1 = val_macro_f1
        best_state = {k: v.clone() for k, v in lp_text_model.state_dict().items()}
    if epoch % 20 == 0:
        print(f"epoch {epoch:3d}  loss={loss.item():.4f}  val_macro_f1={val_macro_f1:.4f}")

lp_text_model.load_state_dict(best_state)
print(f"\nbest val macro-F1: {best_val_f1:.4f}")

In [ ]:
lp_text_model.eval()
with torch.no_grad():
    test_preds = lp_text_model(Xi_test).argmax(dim=1).cpu().numpy()

acc = accuracy_score(y_test_idx, test_preds)
precision, recall, f1_per_class, support = precision_recall_fscore_support(
    y_test_idx, test_preds, labels=range(NUM_CLASSES), average=None, zero_division=0)
macro_f1 = f1_per_class.mean()

print(f"LP+Text (full train) -- Test accuracy: {acc:.4f}   Macro-F1: {macro_f1:.4f}\n")
print("Reference -- Phase 1 (frozen image-only linear): acc=0.3063  macro_f1=0.2469")
print("Reference -- Phase 3 (image+metadata): acc=0.3188  macro_f1=0.2174")
print("Reference -- Contrastive LoRA 30 epochs: acc=0.3594  macro_f1=0.3114\n")

for c, p, r, f, s in zip(classes_sorted, precision, recall, f1_per_class, support):
    print(f"  {c:20s}  P={p:.3f}  R={r:.3f}  F1={f:.3f}  n={s}")

print("\nLearned per-class text multipliers (near-zero means text contributed little for that class):")
for c, m in zip(classes_sorted, lp_text_model.multiplier.detach().cpu().numpy()):
    print(f"  {c:20s}  multiplier={m:.3f}")

pd.DataFrame([
    {"representation": "frozen_image_only_phase1", "accuracy": 0.3063, "macro_f1": 0.2469},
    {"representation": "lp_plus_text", "accuracy": acc, "macro_f1": macro_f1},
]).to_csv("lp_text_fulltrain_results.csv", index=False)

## Few-shot evaluation (the paper's actual focus -- 1/3/5-shot, per Phase 2's setup)
Trains a fresh LP+Text model on ONLY the k support examples per class, evaluated on the full test set. Same shots/seeds as Phase 2/3 for direct comparison.

In [ ]:
SHOTS = [1, 3, 5]
SEEDS = [0, 1, 2, 3, 4]

def sample_support_set(X, y, classes, k, rng):
    """Sample k examples per class -> concatenated (X_support, y_support)."""
    idxs = []
    for c in classes:
        class_idx = np.where(y == c)[0]
        n_take = min(k, len(class_idx))
        idxs.extend(rng.choice(class_idx, size=n_take, replace=False))
    idxs = np.array(idxs)
    return X[idxs], y[idxs]

def train_lp_text(X_support, y_support, n_epochs=200, lr=1e-2):
    """Fresh LP+Text model trained on a small k-shot support set only."""
    m = LPText(IMG_DIM, NUM_CLASSES, text_embeddings_norm).to(DEVICE)
    opt = torch.optim.Adam(m.parameters(), lr=lr, weight_decay=1e-3)  # higher weight_decay -- tiny support set, prone to overfit
    Xs = to_tensor(X_support)
    ys = to_tensor(y_support, dtype=torch.long)
    ce = nn.CrossEntropyLoss()  # no class weighting needed -- support set is already balanced by construction
    for _ in range(n_epochs):
        m.train()
        opt.zero_grad()
        loss = ce(m(Xs), ys)
        loss.backward(); opt.step()
    return m

fewshot_results = []
for k in SHOTS:
    for seed in SEEDS:
        rng = np.random.default_rng(seed)
        X_sup, y_sup = sample_support_set(X_train_img, y_train_idx, range(NUM_CLASSES), k, rng)
        m = train_lp_text(X_sup, y_sup)
        m.eval()
        with torch.no_grad():
            preds = m(Xi_test).argmax(dim=1).cpu().numpy()
        acc_k = accuracy_score(y_test_idx, preds)
        _, _, f1_k, _ = precision_recall_fscore_support(
            y_test_idx, preds, labels=range(NUM_CLASSES), average=None, zero_division=0)
        fewshot_results.append({"shot": k, "seed": seed, "accuracy": acc_k, "macro_f1": f1_k.mean()})

fewshot_df = pd.DataFrame(fewshot_results)
fewshot_summary = fewshot_df.groupby("shot").agg(
    accuracy_mean=("accuracy", "mean"), accuracy_std=("accuracy", "std"),
    macro_f1_mean=("macro_f1", "mean"), macro_f1_std=("macro_f1", "std"),
).reset_index()
print(fewshot_summary.to_string(index=False))
fewshot_summary.to_csv("lp_text_fewshot_results.csv", index=False)

print("\nReference -- Phase 2 image-only prototype few-shot:")
print("  1-shot: acc=0.1544 macro_f1=0.1273")
print("  3-shot: acc=0.1394 macro_f1=0.1091")
print("  5-shot: acc=0.1331 macro_f1=0.0978")
print("\nReference -- Phase 3 image+metadata prototype few-shot:")
print("  1-shot: acc=0.1781 macro_f1=0.1346")
print("  3-shot: acc=0.2663 macro_f1=0.1806")
print("  5-shot: acc=0.2294 macro_f1=0.1479")

## Interpretation notes
- Compare full-train LP+Text against Phase 1/3 and the contrastive LoRA result -- this tells you whether the paper's specific blending method beats what you already tried, using the exact same frozen embeddings (no LoRA retraining needed, much cheaper to test).
- Compare few-shot LP+Text against Phase 2 (image-only) and Phase 3 (image+metadata prototypes) -- the paper's central claim is that this method should be competitive with much more complex few-shot adapters.
- The printed per-class multipliers show which classes' predictions actually leaned on text similarity vs. relied mostly on the image linear probe -- near-zero multipliers for a class mean text wasn't useful for it here.
- The MICCAI reviewers noted all methods stayed below 65% accuracy even at 16-shot on their radiology benchmarks -- so a modest number here is consistent with the literature, not a sign this implementation is broken.